# 24 - ASR + Tashkeel Pipeline (Inference Only)

**Approach**: Combine ASR + Tashkeel for best results
- **ASR**: Use existing seamless_m4t predictions (or transcribe fresh)
- **Tashkeel**: Apply Fine-Tashkeel for better diacritization

**Pipeline**:
1. Audio → seamless_m4t (ASR) → Arabic text
2. Arabic text → Fine-Tashkeel → Diacritized text

**Tasks**:
1. Dev Set: Load ASR + Apply Tashkeel + Evaluate
2. Blind Test: Create Submission

In [ ]:
# Install dependencies
!pip install -q transformers torch accelerate tqdm librosa soundfile sentencepiece pyarabic

In [ ]:
# Setup
import os, sys, json, re, torch, zipfile
from datetime import datetime
from tqdm import tqdm
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings
warnings.filterwarnings('ignore')

IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))

# Paths
DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
TEST_DIR = PROJECT_DIR / 'Test'
TEST_INPUT = TEST_DIR / 'test_input.json'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
SUBMISSION_DIR = PROJECT_DIR / 'submissions'

OUTPUT_DIR.mkdir(exist_ok=True)
SUBMISSION_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load Fine-Tashkeel for diacritization
print("Loading Fine-Tashkeel...")
TASHKEEL_MODEL = 'basharalrfooh/Fine-Tashkeel'
tashkeel_tokenizer = AutoTokenizer.from_pretrained(TASHKEEL_MODEL)
tashkeel_model = AutoModelForSeq2SeqLM.from_pretrained(
    TASHKEEL_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
tashkeel_model.eval()
print(f"Fine-Tashkeel loaded: {sum(p.numel() for p in tashkeel_model.parameters()):,} params")

In [ ]:
# Load data
with open(DEV_INPUT, 'r', encoding='utf-8') as f:
    dev_input = json.load(f)
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f:
    dev_output = json.load(f)
with open(TEST_INPUT, 'r', encoding='utf-8') as f:
    test_input = json.load(f)

print(f"Dev samples: {len(dev_input)}")
print(f"Test samples: {len(test_input)}")

In [ ]:
# Diacritization function
import re

DIACRITIC_PATTERN = re.compile(r'[\u064B-\u0652]')
ARABIC_LETTER_PATTERN = re.compile(r'[\u0621-\u064A]')

def remove_diacritics(text):
    """Remove all diacritics from text."""
    return DIACRITIC_PATTERN.sub('', text)

@torch.inference_mode()
def diacritize_text(text):
    """Add diacritics to undiacritized Arabic text."""
    if not text or not text.strip():
        return text
    
    # Clean input - remove existing diacritics
    clean_text = remove_diacritics(text)
    
    inputs = tashkeel_tokenizer(clean_text, return_tensors="pt", padding=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    outputs = tashkeel_model.generate(
        **inputs,
        max_length=512,
        num_beams=4,
        early_stopping=True
    )
    
    result = tashkeel_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result.strip()

In [ ]:
# Check if seamless_m4t predictions exist
ASR_PREDICTIONS_FILE = OUTPUT_DIR / 'seamless_m4t_dev_predictions.json'
USE_ASR = ASR_PREDICTIONS_FILE.exists()

if USE_ASR:
    print("Loading existing seamless_m4t ASR predictions...")
    with open(ASR_PREDICTIONS_FILE, 'r', encoding='utf-8') as f:
        asr_predictions = json.load(f)
    asr_lookup = {p['id']: p['text_diacritized'] for p in asr_predictions}
    print(f"Loaded {len(asr_lookup)} ASR predictions")
else:
    print("No ASR predictions found - will use input text")
    asr_lookup = {}

In [ ]:
# Metrics - Diac-style evaluation
from diac_evaluation import compute_metrics, print_metrics_table

print("Diac-style evaluation loaded!")

## Dev Set Inference

In [ ]:
# Process dev set
MODEL_KEY = 'asr_tashkeel'

CHECKPOINT_FILE = OUTPUT_DIR / f'{MODEL_KEY}_dev_checkpoint.json'

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_checkpoint(predictions):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({
            'processed_ids': [p['id'] for p in predictions],
            'predictions': predictions
        }, f, ensure_ascii=False)

processed_ids, dev_predictions = load_checkpoint()
print(f"Dev: {len(dev_input)} samples, {len(processed_ids)} already done")

for item in tqdm(dev_input, desc="Dev Set"):
    if item['id'] in processed_ids:
        continue
    
    # Get undiacritized text
    if USE_ASR and item['id'] in asr_lookup:
        # Use ASR output (remove diacritics first, then re-diacritize)
        text_input = remove_diacritics(asr_lookup[item['id']])
    else:
        # Use input text
        text_input = item['text_undiacritized']
    
    # Apply Tashkeel
    try:
        diacritized = diacritize_text(text_input)
    except Exception as e:
        diacritized = text_input  # fallback
    
    dev_predictions.append({'id': item['id'], 'text_diacritized': diacritized})
    save_checkpoint(dev_predictions)

print(f"\nProcessed {len(dev_predictions)} dev samples")

In [ ]:
# Evaluate dev set
print("="*60)
print("DEV SET RESULTS - ASR + Tashkeel Pipeline")
print("="*60)

metrics = compute_metrics(dev_predictions, dev_output)
print_metrics_table(metrics, "DEV SET - ASR + Tashkeel Pipeline")

# Save predictions
with open(OUTPUT_DIR / f'{MODEL_KEY}_dev_predictions.json', 'w', encoding='utf-8') as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=2)

## Test Set Inference

In [ ]:
# Check if seamless_m4t test predictions exist
ASR_TEST_FILE = OUTPUT_DIR / 'seamless_m4t_test_predictions.json'
USE_ASR_TEST = ASR_TEST_FILE.exists()

if USE_ASR_TEST:
    print("Loading seamless_m4t test predictions...")
    with open(ASR_TEST_FILE, 'r', encoding='utf-8') as f:
        asr_test = json.load(f)
    asr_test_lookup = {p['id']: p['text_diacritized'] for p in asr_test}
    print(f"Loaded {len(asr_test_lookup)} test predictions")
else:
    print("No ASR test predictions - will need to run ASR first!")
    asr_test_lookup = {}

In [ ]:
# Process test set
TEST_CHECKPOINT = OUTPUT_DIR / f'{MODEL_KEY}_test_checkpoint.json'

def load_test_checkpoint():
    if TEST_CHECKPOINT.exists():
        with open(TEST_CHECKPOINT, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_test_checkpoint(predictions):
    with open(TEST_CHECKPOINT, 'w', encoding='utf-8') as f:
        json.dump({
            'processed_ids': [p['id'] for p in predictions],
            'predictions': predictions
        }, f, ensure_ascii=False)

test_processed, test_predictions = load_test_checkpoint()
print(f"Test: {len(test_input)} samples, {len(test_processed)} already done")

for item in tqdm(test_input, desc="Test Set"):
    if item['id'] in test_processed:
        continue
    
    # Get undiacritized text
    if USE_ASR_TEST and item['id'] in asr_test_lookup:
        text_input = remove_diacritics(asr_test_lookup[item['id']])
    else:
        text_input = item['text_undiacritized']
    
    # Apply Tashkeel
    try:
        diacritized = diacritize_text(text_input)
    except:
        diacritized = text_input
    
    test_predictions.append({'id': item['id'], 'text_diacritized': diacritized})
    save_test_checkpoint(test_predictions)

print(f"\nProcessed {len(test_predictions)} test samples")

In [ ]:
# Create submission
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
json_file = SUBMISSION_DIR / f'{MODEL_KEY}_submission.json'
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

zip_file = SUBMISSION_DIR / f'{MODEL_KEY}_submission_{timestamp}.zip'
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')

print(f"SUBMISSION READY: {zip_file}")
print(f"Size: {zip_file.stat().st_size/1024:.1f} KB")
print(f"Samples: {len(test_predictions)}")

In [ ]:
# Save predictions
with open(OUTPUT_DIR / f'{MODEL_KEY}_test_predictions.json', 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

print("Predictions saved!")